# Pytorch基礎

In [ ]:
import torch

X = torch.tensor([[1,2,3],[4,5,6]]);
print(X)

tensor([[1, 2, 3],
        [4, 5, 6]])


如果用不同型態的值初始化tensor，pytorch會選擇最通用的型態，也可以在初始化時指定資料型態

In [ ]:
print(X.shape)
print(X.dtype)
print(X[0,1])#存取矩陣第0行地第1列的單一元素
print(X[:,1])#每一列的第1行

torch.Size([2, 3])
torch.int64
tensor(2)
tensor([2, 5])


也可以做運算操作

In [ ]:
print(10*(X+1.0))#加法乘法
print(X.exp())
print(X.float().mean())
print(X.max(dim = 0))
print(X @ X.T)

tensor([[20., 30., 40.],
        [50., 60., 70.]])
tensor([[  2.7183,   7.3891,  20.0855],
        [ 54.5981, 148.4132, 403.4288]])
tensor(3.5000)
torch.return_types.max(
values=tensor([4, 5, 6]),
indices=tensor([1, 1, 1]))
tensor([[14, 32],
        [32, 77]])


可以使用numpy把tensor轉成numpy陣列

In [ ]:
import numpy as np
print(X.numpy())
torch.tensor(np.array([[1,2,3],[4,5,6]]))

[[1 2 3]
 [4 5 6]]


tensor([[1, 2, 3],
        [4, 5, 6]])

呼叫torch.tensor()的時候，最好明確指定dtype = torch.float32，因為32-bit在電腦做深度學習的效果比較好，也可以這樣

In [ ]:
print(torch.FloatTensor([[1.,2.,3.],[2.,3.,6.]]))

tensor([[1., 2., 3.],
        [2., 3., 6.]])


In [ ]:
X[:,1] = -99
print(X)

tensor([[  1, -99,   3],
        [  4, -99,   6]])


也可以使用API來直接對張量做運算

In [ ]:
print(X.relu_())

tensor([[1, 0, 3],
        [4, 0, 6]])


**硬體加速**


---

tensor可以複製到GPU上做執行

In [ ]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(device)

cuda


在GPU上建立tensor，to()可以把tensor轉移到GPU

In [ ]:
m = torch.tensor([[1.,2.,3,],[1.,2.,3.]])
m.to(device)

tensor([[1., 2., 3.],
        [1., 2., 3.]], device='cuda:0')

In [ ]:
print(m.device)

cpu


**Autograd**


---


In [ ]:
x = torch.tensor(4.0,requires_grad = True)
f = x**2
f.backward()
print(x.grad)

tensor(8.)


在每次做前向傳播的時候，pytorch立即建立計算圖，每次算完梯度都會需要做梯度下降，那就要把梯度下降關掉，因為梯度下降不能被放到計算圖裡面

In [ ]:
lr = 0.1
with torch.no_grad():
  x -= lr * x.grad

在跑完一次前向 + 反向 + 梯度下降的時候，要記得把x_grad屬性做歸0，因為先前向傳播，然後反向傳播計算梯度，更新梯度，然後下一次會有新的grad屬性，所以目前的grad屬性跟下一次的反向傳播沒有半毛關係!

In [ ]:
lr = 0.1
x = torch.tensor(5.0,requires_grad = True)

for iteration in range(100):
  f = x**2 #前向
  f.backward() #反向
  with torch.no_grad():
    x -= lr * x.grad
  x.grad.zero_()

print(x)

tensor(1.0185e-09, requires_grad=True)


# 實作線性回歸

In [ ]:
import pandas as pd
from sklearn.datasets import fetch_california_housing

# 1. 載入資料
california = fetch_california_housing()

# 2. 將資料轉換成 Pandas 表格方便觀看
df = pd.DataFrame(california.data, columns=california.feature_names)

# 3. 看看資料長怎樣！
print("--- 前五筆資料長相 ---")
display(df.head()) # 在 Jupyter Notebook/Colab 中，用 display 會比 print 漂亮很多

print("\n--- 資料的形狀大小 ---")
print(df.shape)

--- 前五筆資料長相 ---


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25



--- 資料的形狀大小 ---
(20640, 8)


In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

housing = fetch_california_housing()
X_train,X_test,y_train,y_test = train_test_split(
    housing.data,housing.target,random_state = 42)

X_train = torch.FloatTensor(X_train)
X_test = torch.FloatTensor(X_test)

mean = X_train.mean(dim = 0)
std = X_train.std(dim = 0)

X_train = (X_train - mean)/std
X_test = (X_test - mean)/std

y_train = torch.FloatTensor(y_train).reshape(-1,1)
y_test = torch.FloatTensor(y_test).reshape(-1,1)

In [ ]:
torch.manual_seed(42)
n_features = X_train.shape[1]

w = torch.randn((n_features,1),requires_grad = True)
b = torch.randn(1,requires_grad = True)


lr = 0.4
n_epochs = 20
for epoch in range(n_epochs):
  y_pred = X_train @ w + b
  loss = torch.mean((y_pred - y_train)**2)
  loss.backward()
  with torch.no_grad():
    b -= lr * b.grad;
    w -= lr * w.grad;
    b.grad.zero_()
    w.grad.zero_()
  print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {loss.item()}")

Epoch 1/20, Loss: 14.314901351928711
Epoch 2/20, Loss: 4.6079206466674805
Epoch 3/20, Loss: 2.1211767196655273
Epoch 4/20, Loss: 1.2567073106765747
Epoch 5/20, Loss: 0.928212583065033
Epoch 6/20, Loss: 0.7923619747161865
Epoch 7/20, Loss: 0.728643000125885
Epoch 8/20, Loss: 0.6932001113891602
Epoch 9/20, Loss: 0.6696736216545105
Epoch 10/20, Loss: 0.6517977118492126
Epoch 11/20, Loss: 0.6370760798454285
Epoch 12/20, Loss: 0.6244449615478516
Epoch 13/20, Loss: 0.613395631313324
Epoch 14/20, Loss: 0.603642463684082
Epoch 15/20, Loss: 0.5949961543083191
Epoch 16/20, Loss: 0.5873134136199951
Epoch 17/20, Loss: 0.5804771780967712
Epoch 18/20, Loss: 0.5743879079818726
Epoch 19/20, Loss: 0.5689589977264404
Epoch 20/20, Loss: 0.5641148686408997


In [ ]:
import torch.nn as nn
torch.manual_seed(42)
model = nn.Linear(in_features = n_features,out_features = 1)

print(model.weight)
print(model.bias)

optimizer = torch.optim.SGD(model.parameters(),lr = lr)
loss_mse = nn.MSELoss()

Parameter containing:
tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076]],
       requires_grad=True)
Parameter containing:
tensor([0.3117], requires_grad=True)


In [ ]:
def train_bgd(model, optimizer, criterion, X_train, y_train, n_epochs):
    for epoch in range(n_epochs):
        y_pred = model(X_train)
        loss = criterion(y_pred, y_train)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {loss.item()}")

In [ ]:
train_bgd(model, optimizer, loss_mse, X_train, y_train, n_epochs)

Epoch 1/20, Loss: 4.279625415802002
Epoch 2/20, Loss: 0.7741647958755493
Epoch 3/20, Loss: 0.6209329962730408
Epoch 4/20, Loss: 0.6017913222312927
Epoch 5/20, Loss: 0.5913611650466919
Epoch 6/20, Loss: 0.5830228924751282
Epoch 7/20, Loss: 0.5759017467498779
Epoch 8/20, Loss: 0.5696937441825867
Epoch 9/20, Loss: 0.5642370581626892
Epoch 10/20, Loss: 0.5594235062599182
Epoch 11/20, Loss: 0.5551695227622986
Epoch 12/20, Loss: 0.5514057874679565
Epoch 13/20, Loss: 0.5480729937553406
Epoch 14/20, Loss: 0.5451197028160095
Epoch 15/20, Loss: 0.5425010919570923
Epoch 16/20, Loss: 0.5401774644851685
Epoch 17/20, Loss: 0.5381143093109131
Epoch 18/20, Loss: 0.5362811088562012
Epoch 19/20, Loss: 0.5346510410308838
Epoch 20/20, Loss: 0.5332006216049194


**實作回歸MLP**


---



In [ ]:
import torch.nn as nn

torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(n_features,50),
    nn.ReLU(),
    nn.Linear(50,40),
    nn.ReLU(),
    nn.Linear(40,1)
)

optimizer = torch.optim.SGD(model.parameters(),lr = lr)
train_bgd(model,optimizer,loss_mse,X_train,y_train,n_epochs);

Epoch 1/20, Loss: 4.994747161865234
Epoch 2/20, Loss: 13.083109855651855
Epoch 3/20, Loss: 11.511152267456055
Epoch 4/20, Loss: 1.68071448802948
Epoch 5/20, Loss: 1.307906985282898
Epoch 6/20, Loss: 1.2724119424819946
Epoch 7/20, Loss: 1.22445547580719
Epoch 8/20, Loss: 1.1494942903518677
Epoch 9/20, Loss: 1.035836100578308
Epoch 10/20, Loss: 0.8984470367431641
Epoch 11/20, Loss: 0.7970706224441528
Epoch 12/20, Loss: 0.7516159415245056
Epoch 13/20, Loss: 0.7242521643638611
Epoch 14/20, Loss: 0.7015015482902527
Epoch 15/20, Loss: 0.6818157434463501
Epoch 16/20, Loss: 0.6656752228736877
Epoch 17/20, Loss: 0.6573125123977661
Epoch 18/20, Loss: 0.6801356077194214
Epoch 19/20, Loss: 0.7924781441688538
Epoch 20/20, Loss: 1.3647059202194214


隨機梯度一次只有使用參數來做梯度下降，現在是小批次梯度下降

In [ ]:
from torch.utils.data import TensorDataset,DataLoader


train_dataset = TensorDataset(X_train,y_train)
train_loader = DataLoader(train_dataset,batch_size = 32,shuffle = True)


torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(n_features,50),
    nn.ReLU(),
    nn.Linear(50,40),
    nn.ReLU(),
    nn.Linear(40,1)
)
model = model.to(device)

def train(model,optimizer,criterion,train_loader,n_epochs):
  model.train()#把模型切換到訓練模式
  for epoch in range(n_epochs):
    total_loss = 0;
    for x_batch,y_batch in train_loader:
      x_batch = x_batch.to(device)#cuda
      y_batch = y_batch.to(device)#cuda
      loss = criterion(model(x_batch),y_batch)
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()
      total_loss += loss.item()
    print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {total_loss / len(train_loader)}")

train(model,optimizer,loss_mse,train_loader,n_epochs)

Epoch 1/20, Loss: 4.994967253740168
Epoch 2/20, Loss: 4.994416230473637
Epoch 3/20, Loss: 4.99407634114431
Epoch 4/20, Loss: 4.994189716075078
Epoch 5/20, Loss: 4.994873687255481
Epoch 6/20, Loss: 4.994905912679089
Epoch 7/20, Loss: 4.994786454626351
Epoch 8/20, Loss: 4.995251296472944
Epoch 9/20, Loss: 4.994527478848607
Epoch 10/20, Loss: 4.9954706627475325
Epoch 11/20, Loss: 4.994876146809129
Epoch 12/20, Loss: 4.996293439845409
Epoch 13/20, Loss: 4.993915105161588
Epoch 14/20, Loss: 4.994992249268146
Epoch 15/20, Loss: 4.995854025044717
Epoch 16/20, Loss: 4.99447056183145
Epoch 17/20, Loss: 4.995077268643812
Epoch 18/20, Loss: 4.995002901258547
Epoch 19/20, Loss: 4.994637459270225
Epoch 20/20, Loss: 4.995051372149759


In [ ]:
!pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 27.7 MB/s eta 0:00:00


In [ ]:
import torchmetrics

def evaluate_tm(model, data_loader, metric):
    model.eval() #評估模式
    metric.reset()  # reset the metric at the beginning
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)  # update it at each iteration
    return metric.compute()  # compute the final result at the end

**使用自訂模組來建立Sequential模型**


---

把部分輸出直接連接到輸出層，好處是可以把深層模式以及簡單規則通通學習，不過這不是Sequential的

In [ ]:
import torch.nn as nn
import torch

class WideAndDeep(nn.Module):
    def __init__(self,n_features):
      super().__init__()
      self.deep_stack = nn.Sequential(
          nn.Linear(n_features,50),
          nn.ReLU(),
          nn.Linear(50,40),
          nn.ReLU()
      )
      self.out_layer = nn.Linear(40 + n_features,1)

    def forward(self,x):
      wide_deep_out = self.deep_stack(x)
      # Concatenate the deep path output with the original features (wide path)
      combined_features = torch.cat([wide_deep_out, x], dim=1)
      return self.out_layer(combined_features)

In [ ]:

torch.manual_seed(42)
model = WideAndDeep(n_features)
model = model.to(device)
optimizer = torch.optim.SGD(model.parameters(),lr = lr)
train(model,optimizer,loss_mse,train_loader,n_epochs)

Epoch 1/20, Loss: nan
Epoch 2/20, Loss: nan
Epoch 3/20, Loss: nan
Epoch 4/20, Loss: nan
Epoch 5/20, Loss: nan
Epoch 6/20, Loss: nan
Epoch 7/20, Loss: nan
Epoch 8/20, Loss: nan
Epoch 9/20, Loss: nan
Epoch 10/20, Loss: nan
Epoch 11/20, Loss: nan
Epoch 12/20, Loss: nan
Epoch 13/20, Loss: nan
Epoch 14/20, Loss: nan
Epoch 15/20, Loss: nan
Epoch 16/20, Loss: nan
Epoch 17/20, Loss: nan
Epoch 18/20, Loss: nan
Epoch 19/20, Loss: nan
Epoch 20/20, Loss: nan


如果要把特徵分成兩個部分，一部分送入wide路徑。一部分送入deep路徑，在forward裡面把輸入分開就可以了，不過在init的架構部分要改改

In [ ]:
import torch.nn as nn
import torch

class WideAndDeepV2(nn.Module):
  def __init__(self,n_features):
    super().__init__()
    self.deep_stack = nn.Sequential(
        nn.Linear(n_features-2,50),
        nn.ReLU(),
        nn.Linear(50,40),
        nn.ReLU()
        #deep part output has 40:)
    )
    self.output = nn.Linear(40+5,dim = 1)

    def forward(self,X):
      X_wide = X[:,:5]
      X_deep = X[:,2:]
      deep_output = self.deep_stack(X_deep)
      deep_wide_output = torch.cat([deep_output,X_wide],dim = 1)
      return self.out_layer(deep_wide_output)


但是這樣的模型，在特徵分開時的wide和deep部分有部分重疊，我們希望wide跟deep 可以獨立

In [ ]:
import torch.nn as nn
import torch
from torch.utils.data import TensorDataset, DataLoader

# Define device
device = "cuda" if torch.cuda.is_available() else "cpu"

class WideAndDeepV3(nn.Module):
  def __init__(self, n_features):
    super().__init__()
    self.deep_stack = nn.Sequential(
        nn.Linear(n_features - 2, 50),
        nn.ReLU(),
        nn.Linear(50, 40),
        nn.ReLU(),
        nn.Linear(40, 30),
        nn.ReLU()
    )
    self.output = nn.Linear(30 + 5, out_features=1)

  def forward(self, X_deep, X_wide):
    deep_output = self.deep_stack(X_deep)
    deep_wide_output = torch.cat([deep_output, X_wide], dim=1)
    return self.output(deep_wide_output) # Updated from out_layer to output

# train data preparation
train_data_wd = TensorDataset(X_train[:, 2:], X_train[:, :5], y_train)
train_loader_wd = DataLoader(train_data_wd, batch_size=32, shuffle=True)

def train_multi_in(model, optimizer, criterion, train_loader, n_epochs):
    model.train()
    for epoch in range(n_epochs):
        total_loss = 0
        for *X_batch, y_batch in train_loader:
          X_batch = [x.to(device) for x in X_batch]
          y_batch = y_batch.to(device)
          optimizer.zero_grad()
          loss = criterion(model(*X_batch), y_batch)
          loss.backward()
          optimizer.step()
          total_loss += loss.item()
        print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {total_loss / len(train_loader)}")

torch.manual_seed(42)
learning_rate = 0.01
model = WideAndDeepV3(n_features).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0)
mse = nn.MSELoss()
train_multi_in(model, optimizer, mse, train_loader_wd, n_epochs)

Epoch 1/20, Loss: 0.7677398461017234
Epoch 2/20, Loss: 0.5073233988474716
Epoch 3/20, Loss: 0.46756654560627525
Epoch 4/20, Loss: 0.4496093529376609
Epoch 5/20, Loss: 0.43495650737246205
Epoch 6/20, Loss: 0.42740413511156544
Epoch 7/20, Loss: 0.410770622443808
Epoch 8/20, Loss: 0.399785363908149
Epoch 9/20, Loss: 0.38844558124022543
Epoch 10/20, Loss: 0.3741517513374652
Epoch 11/20, Loss: 0.3673122948543592
Epoch 12/20, Loss: 0.36032125464648257
Epoch 13/20, Loss: 0.3577947949852086
Epoch 14/20, Loss: 0.35253656951110224
Epoch 15/20, Loss: 0.34699854090009347
Epoch 16/20, Loss: 0.3494430348307879
Epoch 17/20, Loss: 0.3440493436699564
Epoch 18/20, Loss: 0.34020739935400074
Epoch 19/20, Loss: 0.33940006954179813
Epoch 20/20, Loss: 0.3368693328932051


In [ ]:
import torch.nn as nn
import torch
from torch.utils.data import TensorDataset, DataLoader

# Define device if not already defined
device = "cuda" if torch.cuda.is_available() else "cpu"

class WideAndDeepV4(nn.Module): # Fixed: nn.Module instead of nn.model
  def __init__(self, n_features):
    super().__init__()
    self.deep_stack = nn.Sequential(
        nn.Linear(n_features - 2, 50), nn.ReLU(),
        nn.Linear(50, 40), nn.ReLU(),
        nn.Linear(40, 30), nn.ReLU()
    )
    self.output = nn.Linear(30 + 5, 1)
    self.aux_output = nn.Linear(30, 1)

  def forward(self, X_deep, X_wide):
    deep = self.deep_stack(X_deep)
    deep_wide = torch.cat([deep, X_wide], dim=1)
    deep_wide_main_output = self.output(deep_wide)
    aux_deep_output = self.aux_output(deep)
    return deep_wide_main_output, aux_deep_output

def train_multi_out(model, optimizer, criterion, train_loader, n_epochs):
    model.train()
    for epoch in range(n_epochs):
        total_loss = 0
        for X_deep, X_wide, y_batch in train_loader:
            X_deep, X_wide, y_batch = X_deep.to(device), X_wide.to(device), y_batch.to(device)

            optimizer.zero_grad()
            y_pred, y_aux_pred = model(X_deep, X_wide)

            main_loss = criterion(y_pred, y_batch)
            aux_loss = criterion(y_aux_pred, y_batch)
            loss = 0.8 * main_loss + 0.2 * aux_loss

            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {total_loss / len(train_loader):.4f}")

torch.manual_seed(42)
learning_rate = 0.01

model = WideAndDeepV4(n_features).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0)
mse = nn.MSELoss()

# train data preparation and DataLoader
train_data_wd = TensorDataset(X_train[:, 2:], X_train[:, :5], y_train)
train_loader_wd = DataLoader(train_data_wd, batch_size=32, shuffle=True)

train_multi_out(model, optimizer, mse, train_loader_wd, n_epochs)

Epoch 1/20, Loss: 0.9627
Epoch 2/20, Loss: 0.6005
Epoch 3/20, Loss: 0.5290
Epoch 4/20, Loss: 0.4931
Epoch 5/20, Loss: 0.4688
Epoch 6/20, Loss: 0.4539
Epoch 7/20, Loss: 0.4394
Epoch 8/20, Loss: 0.4241
Epoch 9/20, Loss: 0.4103
Epoch 10/20, Loss: 0.3916
Epoch 11/20, Loss: 0.3835
Epoch 12/20, Loss: 0.3753
Epoch 13/20, Loss: 0.3703
Epoch 14/20, Loss: 0.3731
Epoch 15/20, Loss: 0.3651
Epoch 16/20, Loss: 0.3593
Epoch 17/20, Loss: 0.3575
Epoch 18/20, Loss: 0.3599
Epoch 19/20, Loss: 0.3532
Epoch 20/20, Loss: 0.3486
